In [62]:
import sqlite3
import pandas as pd
import plotly.express as px
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load unique books with genres and publication date
query = """
SELECT DISTINCT title, genres, publication_date
FROM books 
WHERE genres IS NOT NULL AND genres != ''
  AND publication_date IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
df = df.drop_duplicates(subset=['title'])

# Extract decade 
def get_decade(date_str):
    try:
        year = pd.to_datetime(date_str, errors='coerce').year
        if pd.isna(year):
            return None
        decade = (year // 10) * 10
        return f"{decade}s" if decade <= 2020 else None
    except:
        return None

df['decade'] = df['publication_date'].apply(get_decade)
df = df.dropna(subset=['decade'])

# Clean genre name
def clean_genre(genre_str):
    genre = genre_str.lower().strip()
    genre = re.sub(r'[^a-z0-9\s&]', '', genre)
    genre = re.sub(r'\s+', ' ', genre).strip()
    genre = ' '.join(word.capitalize() for word in genre.split())
    return genre

# Nonfiction keywords (pure)
nonfiction_keywords = {
    'nonfiction', 'non-fiction', 'biography', 'memoir', 'history', 'philosophy',
    'religion', 'science', 'education', 'psychology', 'self-help', 'business',
    'economics', 'politics', 'law', 'health', 'sports', 'travel', 'cooking',
    'art', 'music', 'photography', 'true crime'
}

# Expand rows
rows = []
for _, row in df.iterrows():
    raw_genres = [g.strip() for g in row['genres'].split(',') if g.strip()]
    cleaned_genres = [clean_genre(g) for g in raw_genres if clean_genre(g)]
    if not cleaned_genres:
        continue
    # Pure nonfiction check
    if all(g.lower() in nonfiction_keywords for g in cleaned_genres):
        rows.append({'decade': row['decade'], 'genre': 'NonFiction (pure)'})
    # Add each individual genre (will later remove generic nonfiction)
    for g in cleaned_genres:
        rows.append({'decade': row['decade'], 'genre': g})

df_expanded = pd.DataFrame(rows)

df_expanded = df_expanded[~df_expanded['genre'].str.lower().isin(['nonfiction', 'non-fiction'])]

# Count and cumulative sum
counts = df_expanded.groupby(['decade', 'genre']).size().reset_index(name='count')
counts['decade_num'] = counts['decade'].str.extract(r'(\d+)')[0].astype(int)
counts = counts.sort_values(['genre', 'decade_num'])
counts['cumulative'] = counts.groupby('genre')['count'].cumsum()

# Keep top 15 genres by final cumulative value
final_cum = counts.groupby('genre')['cumulative'].max().nlargest(15).index
counts_top = counts[counts['genre'].isin(final_cum)]

# Plot
fig1 = px.line(counts_top, x='decade', y='cumulative', color='genre',
               title='Cumulative Book Count per Decade (Top 15 Genres)',
               labels={'decade': 'Decade', 'cumulative': 'Cumulative Number of Books'},
               markers=True)

fig1.update_layout(
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=500,
    xaxis=dict(tickangle=-45, gridcolor='#444'),
    yaxis=dict(gridcolor='#444'),
    legend=dict(x=1.02, y=1, bgcolor='rgba(0,0,0,0.5)')
)

# Save
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '1_genre_cumulative_timeline.html')
fig1.write_html(output_path)

conn.close()

In [63]:
 fig1.show()

In [64]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load books that have historical_events or historical_figures 
query = """
SELECT title, rating, ratings_count, historical_events, historical_figures
FROM books 
WHERE (historical_events IS NOT NULL AND historical_events != '[]')
   OR (historical_figures IS NOT NULL AND historical_figures != '[]')
"""
df = pd.read_sql_query(query, conn)

# Helper to safely parse JSON arrays (if stored as JSON string)
def parse_list(val):
    if not val or val == '[]':
        return []
    try:
        return json.loads(val)
    except:
        # j split by comma if it's a plain string
        return [x.strip() for x in str(val).split(',') if x.strip()]

# Extract events and figures
events_list = []
figures_list = []
for _, row in df.iterrows():
    events = parse_list(row['historical_events'])
    for e in events:
        events_list.append({'name': e, 'rating': row['rating']})
    figures = parse_list(row['historical_figures'])
    for f in figures:
        figures_list.append({'name': f, 'rating': row['rating']})

df_events = pd.DataFrame(events_list)
df_figures = pd.DataFrame(figures_list)

# count and average rating
def agg_data(df):
    if df.empty:
        return pd.DataFrame(columns=['name', 'count', 'avg_rating'])
    grouped = df.groupby('name').agg(
        count=('name', 'size'),
        avg_rating=('rating', 'mean')
    ).reset_index()
    grouped = grouped.sort_values('count', ascending=False).head(20)  # top 20
    return grouped

events_agg = agg_data(df_events)
figures_agg = agg_data(df_figures)

# Create figure with two traces 
fig2 = go.Figure()

# Trace for events
fig2.add_trace(go.Bar(
    y=events_agg['name'],
    x=events_agg['count'],
    orientation='h',
    name='Historical Events',
    marker_color='#9b59b6',
    text=events_agg['count'],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Books: %{x}<br>Avg rating: %{customdata:.2f}<extra></extra>',
    customdata=events_agg['avg_rating'],
    visible=True
))

# Trace for figures (initially hidden)
fig2.add_trace(go.Bar(
    y=figures_agg['name'],
    x=figures_agg['count'],
    orientation='h',
    name='Historical Figures',
    marker_color='#3498db',
    text=figures_agg['count'],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Books: %{x}<br>Avg rating: %{customdata:.2f}<extra></extra>',
    customdata=figures_agg['avg_rating'],
    visible=False
))

# Dropdown buttons
buttons = [
    dict(label='Historical Events', method='update',
         args=[{'visible': [True, False]},
               {'title': 'Top Historical Events by Number of Books'}]),
    dict(label='Historical Figures', method='update',
         args=[{'visible': [False, True]},
               {'title': 'Top Historical Figures by Number of Books'}])
]

fig2.update_layout(
    title='Historical Events and Figures in Books',
    xaxis_title='Number of Books',
    yaxis_title='',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=600,
    margin=dict(l=150, r=30, t=80, b=50),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.98,                
        y=0.98,                
        xanchor='right',       
        yanchor='top',         
        bgcolor='#333',
        font=dict(color='white'),
        bordercolor='#666',
        borderwidth=1
    )]
)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '2_historical_events_figures.html')
fig2.write_html(output_path)

conn.close()

In [65]:
fig2.show()

In [9]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import json
import re

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load data
query = """
SELECT historical_events, rating, ratings_count
FROM books 
WHERE historical_events IS NOT NULL AND historical_events != ''
  AND rating IS NOT NULL
  AND ratings_count IS NOT NULL
"""
df = pd.read_sql_query(query, conn)

# Helper to parse events list
def parse_events(val):
    if not val or val == '[]':
        return []
    try:
        events = json.loads(val)
        if isinstance(events, list):
            return [str(e).strip() for e in events if str(e).strip()]
    except:
        return [v.strip() for v in str(val).split(',') if v.strip()]
    return []

# Normalize event names (merge American Civil War with Civil War)
def normalize_event(event_str):
    s = event_str.lower().strip()
    if 'american civil war' in s:
        return 'Civil War'
    if 'civil war' in s:
        return 'Civil War'
    if any(phrase in s for phrase in ['world war ii', 'world war 2', 'wwii', 'ww2', 'second world war']):
        return 'World War II'
    s = re.sub(r'[^\w\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return ' '.join(word.capitalize() for word in s.split())

# Expand rows
rows = []
for _, row in df.iterrows():
    events = parse_events(row['historical_events'])
    if not events:
        continue
    for ev in events:
        ev_norm = normalize_event(ev)
        rows.append({
            'event': ev_norm,
            'rating': row['rating'],
            'ratings_count': row['ratings_count']
        })

df_events = pd.DataFrame(rows)

# Top 10 events by book count
top_events = df_events['event'].value_counts().head(10).index.tolist()
df_top = df_events[df_events['event'].isin(top_events)]

# Aggregate: average rating, TOTAL ratings count (sum)
stats = df_top.groupby('event').agg(
    avg_rating=('rating', 'mean'),
    total_ratings=('ratings_count', 'sum')   
).reset_index()

# Sort for horizontal bar chart (by average rating)
stats = stats.sort_values('avg_rating', ascending=True)

# Create subplots (1 row, 2 columns)
# Create subplots (1 row, 2 columns)
fig3 = make_subplots(rows=1, cols=2,
                     shared_yaxes=True,
                     subplot_titles=('Average Rating', 'Total Number of Ratings'))

# Left subplot: average rating
fig3.add_trace(go.Bar(
    x=stats['avg_rating'],
    y=stats['event'],
    orientation='h',
    marker_color='#9b59b6',
    text=stats['avg_rating'].round(2),
    textposition='outside',
    textfont=dict(size=11),
    name='Avg Rating',
    hovertemplate='Event: %{y}<br>Avg rating: %{x:.2f}<extra></extra>',
    width=0.7
), row=1, col=1)

# Right subplot
fig3.add_trace(go.Bar(
    x=stats['total_ratings'],
    y=stats['event'],
    orientation='h',
    marker_color='#e74c8c',
    text=stats['total_ratings'].apply(lambda x: f'{x:,.0f}'),
    textposition='outside',
    textfont=dict(size=10),
    name='Total Ratings',
    hovertemplate='Event: %{y}<br>Total ratings: %{x:,.0f}<extra></extra>',
    width=0.7
), row=1, col=2)

fig3.update_layout(
    title='Top 10 Historical Events: Average Rating vs. Total Number of Ratings',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=600,
    margin=dict(l=180, r=140, t=80, b=50), 
    showlegend=False
)

# Axes
fig3.update_xaxes(title_text='Average Rating', row=1, col=1, gridcolor='#444', range=[3, 4.2]) 
fig3.update_xaxes(title_text='Total Number of Ratings', row=1, col=2, gridcolor='#444', type='log')
fig3.update_yaxes(title_text='Historical Event', row=1, col=1, gridcolor='#444')
fig3.update_yaxes(matches='y', row=1, col=2)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '3_event_rating_popularity.html')
fig3.write_html(output_path)

conn.close()

In [10]:
fig3.show()

In [76]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load data: rating and publication_date
query = """
SELECT DISTINCT title, rating, publication_date
FROM books 
WHERE rating IS NOT NULL AND rating != 0
  AND publication_date IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
df = df.drop_duplicates(subset=['title'])

# Extract decade
def get_decade(date_str):
    try:
        year = pd.to_datetime(date_str, errors='coerce').year
        if pd.isna(year):
            return None
        decade = (year // 10) * 10
        return decade if decade <= 2020 else None
    except:
        return None

df['decade'] = df['publication_date'].apply(get_decade)
df = df.dropna(subset=['decade'])

# Calculate average rating per decade
decade_avg = df.groupby('decade')['rating'].mean().reset_index()
decade_avg = decade_avg.sort_values('decade')

# Set baseline as the earliest decade's average
baseline = decade_avg.iloc[0]['rating']
baseline_decade = decade_avg.iloc[0]['decade']
decade_avg['change'] = decade_avg['rating'] - baseline

# Prepare data for waterfall chart
decades = decade_avg['decade'].astype(str).tolist()
changes = decade_avg['change'].tolist()
ratings = decade_avg['rating'].tolist()

# Create waterfall bars
fig4 = go.Figure()

# Add bars for change (positive and negative)
fig4.add_trace(go.Bar(
    x=decades,
    y=changes,
    name='Change from baseline',
    marker_color=['#e74c8c' if x < 0 else '#2ecc71' for x in changes],
    text=[f'{c:+.3f}' for c in changes],         
    textposition='outside',
    hovertemplate='Decade: %{x}<br>Change: %{y:+.3f}<br>Avg rating: %{customdata:.3f}<extra></extra>',  
    customdata=ratings
))

# Add a horizontal line at y=0
fig4.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.5)

# Annotate baseline value
fig4.add_annotation(
    x=0.02, y=0.98, xref='paper', yref='paper',
    text=f'Baseline ({baseline_decade}s): {baseline:.3f}',   # changed to 3 decimals
    showarrow=False,
    font=dict(color='white', size=12),
    bgcolor='rgba(0,0,0,0.6)',
    bordercolor='#9b59b6',
    borderwidth=1
)

fig4.update_layout(
    title='Change in Average Book Rating per Decade (relative to earliest decade)',
    xaxis_title='Decade',
    yaxis_title='Change in Average Rating',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=500,
    xaxis=dict(tickangle=-45, gridcolor='#444'),
    yaxis=dict(gridcolor='#444')
)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '4_rating_waterfall.html')
fig4.write_html(output_path)

conn.close()

In [75]:
fig4.show()

In [77]:
import sqlite3
import pandas as pd
import plotly.express as px
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load books with historical events and genres
query = """
SELECT DISTINCT title, genres, historical_events
FROM books 
WHERE historical_events IS NOT NULL AND historical_events != ''
  AND genres IS NOT NULL AND genres != ''
"""
df = pd.read_sql_query(query, conn)
df = df.drop_duplicates(subset=['title'])

# Helper to parse historical events (JSON array or plain string list)
def parse_events(val):
    if not val or val == '[]':
        return []
    try:
        events = json.loads(val)
        if isinstance(events, list):
            return [e.strip() for e in events if e.strip()]
        else:
            return []
    except:
        # fallback: split by comma
        return [e.strip() for e in str(val).split(',') if e.strip()]

# Expand rows
rows = []
for _, row in df.iterrows():
    events = parse_events(row['historical_events'])
    if not events:
        continue
    genres = [g.strip() for g in row['genres'].split(',') if g.strip()]
    for genre in genres:
        for event in events:
            rows.append({
                'title': row['title'],
                'genre': genre,
                'event': event
            })

if not rows:
    print("No events found. Creating empty plot.")
    df_expanded = pd.DataFrame(columns=['title', 'genre', 'event'])
else:
    df_expanded = pd.DataFrame(rows)

# Clean genre and event names
def clean_name(s):
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return ' '.join(word.capitalize() for word in s.split())

df_expanded['genre_clean'] = df_expanded['genre'].apply(clean_name)
df_expanded['event_clean'] = df_expanded['event'].apply(clean_name)

# American Civil War into Civil War
df_expanded['event_clean'] = df_expanded['event_clean'].replace('American Civil War', 'Civil War')

# Keep only top 10 events by total mentions
top_events = df_expanded['event_clean'].value_counts().head(10).index
df_filtered = df_expanded[df_expanded['event_clean'].isin(top_events)]

# Keep only top 10 genres by number of books 
top_genres = df_filtered.groupby('genre_clean')['title'].nunique().nlargest(10).index
df_filtered = df_filtered[df_filtered['genre_clean'].isin(top_genres)]

# compute proportion: for each genre, fraction of its books that mention each event

books_per_genre = df_filtered.groupby('genre_clean')['title'].nunique().reset_index(name='total_books')

# Count pairs – each book counted once per event
event_counts = df_filtered.groupby(['genre_clean', 'event_clean'])['title'].nunique().reset_index(name='book_count')

# Merge to get proportion
merged = event_counts.merge(books_per_genre, on='genre_clean')
merged['proportion'] = (merged['book_count'] / merged['total_books']) * 100

# Pivot for heatmap
pivot = merged.pivot(index='genre_clean', columns='event_clean', values='proportion').fillna(0)

# Create heamap
fig5 = px.imshow(pivot,
                 text_auto='.1f',
                 aspect='auto',
                 title='Heatmap: Genre vs. Historical Event (% of books in genre mentioning event)',
                 labels=dict(x='Historical Event', y='Genre', color='% of books'),
                 color_continuous_scale='Viridis')

fig5.update_layout(
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=600,
    xaxis=dict(tickangle=-45),
    margin=dict(l=100, r=30, t=80, b=100)
)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '5_genre_event_heatmap.html')
fig5.write_html(output_path)

conn.close()

In [78]:
fig5.show()

In [97]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load data: title, author, historical_events
query = """
SELECT DISTINCT title, author, historical_events
FROM books 
WHERE author IS NOT NULL AND author != ''
  AND historical_events IS NOT NULL AND historical_events != ''
"""
df = pd.read_sql_query(query, conn)

# Helper to parse historical_events
def parse_events(val):
    if not val or val == '[]':
        return []
    try:
        events = json.loads(val)
        if isinstance(events, list):
            return [str(e).strip() for e in events if str(e).strip()]
    except:
        return [v.strip() for v in str(val).split(',') if v.strip()]
    return []

# Normalize event names: merge Civil War variants, clean punctuation
def normalize_event(event):
    s = event.lower().strip()
    if 'civil war' in s:
        return 'Civil War'
    s = re.sub(r'[^\w\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return ' '.join(word.capitalize() for word in s.split())

# Expand rows
rows = []
for _, row in df.iterrows():
    events = parse_events(row['historical_events'])
    if not events:
        continue
    for ev in events:
        ev_norm = normalize_event(ev)
        rows.append({
            'author': row['author'],
            'title': row['title'],
            'event': ev_norm
        })

df_expanded = pd.DataFrame(rows)

# Count total books per author 
total_books = df_expanded.groupby('author')['title'].nunique().reset_index(name='total_books')

# For each event, count books per author 
event_author_counts = df_expanded.groupby(['event', 'author'])['title'].nunique().reset_index(name='event_books')

# Get top 10 events by total number of books mentioning the event
event_totals = event_author_counts.groupby('event')['event_books'].sum().nlargest(10).index.tolist()

# Build figure with dropdown
fig6 = go.Figure()

# Prepare dropdown buttons
buttons = []
for event in event_totals:
    # Get top 20 authors for this event
    top_authors = event_author_counts[event_author_counts['event'] == event].nlargest(20, 'event_books')
    top_authors = top_authors.merge(total_books, on='author', how='left')
    top_authors = top_authors.sort_values('event_books', ascending=True)
    authors = top_authors['author'].tolist()
    event_counts = top_authors['event_books'].tolist()
    total_counts = top_authors['total_books'].tolist()
    
    # Build line data with None separators to avoid zigzags
    line_x = []
    line_y = []
    for author, cnt in zip(authors, event_counts):
        line_x.extend([0, cnt, None])
        line_y.extend([author, author, None])
    
    buttons.append(dict(
        label=event,
        method='update',
        args=[
            {
                'x': [event_counts, total_counts, line_x],
                'y': [authors, authors, line_y],
                'mode': ['markers', 'markers', 'lines'],
                'marker': [{'size': 10, 'color': '#e74c8c'}, {'size': 5, 'color': '#ccc'}, {}],
                'line': [{'width': 0}, {'width': 0}, {'color': '#555', 'width': 1}],
                'name': ['Books about event', 'Total books by author', ''],
                'showlegend': [True, True, False]
            },
            {'title': f'Authors writing about "{event}" (top 20 by number of books)'}
        ]
    ))

# Initial traces
fig6.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(size=8, color='#e74c8c'), name='Books about event'))
fig6.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(size=4, color='#ccc'), name='Total books by author'))
fig6.add_trace(go.Scatter(x=[], y=[], mode='lines', line=dict(color='#555', width=1), showlegend=False))

# Dropdown menu positioned below legend
fig6.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.98,             
        y=0.7,                
        xanchor='right',
        yanchor='middle',
        bgcolor='#333',
        font=dict(color='white'),
        bordercolor='#666',
        borderwidth=1
    )],
    title='Select an event from dropdown',
    xaxis_title='Number of books',
    yaxis_title='Author',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=600,
    margin=dict(l=150, r=30, t=80, b=50),
    hovermode='closest'
)

# Set initial data for first event
first_event = event_totals[0]
first_data = event_author_counts[event_author_counts['event'] == first_event].nlargest(20, 'event_books')
first_data = first_data.merge(total_books, on='author')
first_data = first_data.sort_values('event_books', ascending=True)
authors = first_data['author'].tolist()
event_counts = first_data['event_books'].tolist()
total_counts = first_data['total_books'].tolist()
# Build line_x, line_y with None separators
line_x = []
line_y = []
for author, cnt in zip(authors, event_counts):
    line_x.extend([0, cnt, None])
    line_y.extend([author, author, None])

fig6.update_traces(selector=0, x=event_counts, y=authors)
fig6.update_traces(selector=1, x=total_counts, y=authors)
fig6.update_traces(selector=2, x=line_x, y=line_y)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '6_authors_event_lollipop.html')
fig6.write_html(output_path)

conn.close()

In [96]:
fig6.show()

In [21]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import json
import os

# Connect to database
db_path = '../frontend/public/books.db'
conn = sqlite3.connect(db_path)

# Load books with historical events and author country
query = """
SELECT DISTINCT title, author, historical_events, country
FROM books 
WHERE historical_events IS NOT NULL AND historical_events != ''
  AND country IS NOT NULL AND country != ''
"""
df = pd.read_sql_query(query, conn)

# Parse historical_events fiel
def parse_events(val):
    if not val or val == '[]':
        return []
    try:
        events = json.loads(val)
        if isinstance(events, list):
            return [str(e).strip() for e in events if str(e).strip()]
    except:
        return [v.strip() for v in str(val).split(',') if v.strip()]
    return []

# Extract civil war name from event string
def extract_civil_war(event):
    s = str(event).lower()
    patterns = {
        'american': 'American Civil War',
        'us civil war': 'American Civil War',
        'united states civil war': 'American Civil War',
        'spanish': 'Spanish Civil War',
        'russian': 'Russian Civil War',
        'english': 'English Civil War',
        'chinese': 'Chinese Civil War',
        'syrian': 'Syrian Civil War',
        'libyan': 'Libyan Civil War',
        'yemeni': 'Yemeni Civil War',
        'iraqi': 'Iraqi Civil War',
        'afghan': 'Afghan Civil War',
        'nepalese': 'Nepalese Civil War',
        'angolan': 'Angolan Civil War',
        'sri lankan': 'Sri Lankan Civil War',
        'mexican': 'Mexican Civil War',
        'colombian': 'Colombian Civil War',
        'salvadoran': 'Salvadoran Civil War',
        'guatemalan': 'Guatemalan Civil War',
        'nicaraguan': 'Nicaraguan Civil War',
        'cambodian': 'Cambodian Civil War',
        'laotian': 'Laotian Civil War',
        'bangladesh': 'Bangladesh Civil War',
        'pakistani': 'Pakistani Civil War',
        'sudanese': 'Sudanese Civil War',
        'somali': 'Somali Civil War',
        'rwandan': 'Rwandan Civil War',
        'burundian': 'Burundian Civil War',
        'ivorian': 'Ivorian Civil War',
        'sierra leone': 'Sierra Leone Civil War',
        'liberian': 'Liberian Civil War',
        'central african': 'Central African Civil War',
        'congolese': 'Congolese Civil War',
        'ethiopian': 'Ethiopian Civil War',
        'eritrean': 'Eritrean Civil War',
        'mozambican': 'Mozambican Civil War',
        'zimbabwean': 'Zimbabwean Civil War',
    }
    for key, name in patterns.items():
        if key in s:
            return name
    return None

# Build table: book, country, civil war
rows = []
for _, row in df.iterrows():
    events = parse_events(row['historical_events'])
    if not events:
        continue
    for ev in events:
        civil_war = extract_civil_war(ev)
        if civil_war:
            rows.append({
                'title': row['title'],
                'country': row['country'],
                'civil_war': civil_war
            })

df_civil = pd.DataFrame(rows)

if df_civil.empty:
    raise ValueError("No civil war events found in database.")

# Top 10 civil wars by number of unique books
war_totals = df_civil.groupby('civil_war')['title'].nunique().sort_values(ascending=False)
top_wars = war_totals.head(10).index.tolist()

# Add aggregated option "All civil wars"
war_options = ['Civil War (All)'] + top_wars

# Count books per country for each option
war_data = {}
war_data['Civil War (All)'] = (
    df_civil.groupby('country')['title']
    .nunique()
    .reset_index(name='count')
)
for war in top_wars:
    df_war = df_civil[df_civil['civil_war'] == war]
    war_data[war] = (
        df_war.groupby('country')['title']
        .nunique()
        .reset_index(name='count')
    )

# Compute maximum value for each option (used for dynamic color scale)
war_max = {}
for war in war_options:
    df_tmp = war_data[war]
    war_max[war] = df_tmp['count'].max() if not df_tmp.empty else 1

# Create figure
fig11 = go.Figure()

# Add traces for all options (only first visible initially)
for i, war in enumerate(war_options):
    df_counts = war_data[war]
    trace = go.Choropleth(
        locations=df_counts['country'],
        locationmode='country names',
        z=df_counts['count'],
        text=df_counts['country'],
        coloraxis='coloraxis',
        name=war,
        visible=(i == 0),
        hovertemplate='<b>%{text}</b><br>Books: %{z}<extra></extra>'
    )
    fig11.add_trace(trace)

# Build dropdown buttons with dynamic color scale update
buttons = []
for i, war in enumerate(war_options):
    visible = [False] * len(war_options)
    visible[i] = True
    buttons.append(dict(
        label=war,
        method='update',
        args=[
            {'visible': visible},
            {
                'title': f'Books about {war} by country of author',
                'coloraxis.cmax': war_max[war]   # update color scale max
            }
        ]
    ))

# Layout configuration
fig11.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.01,
        y=0.8,
        xanchor='left',
        yanchor='top',
        bgcolor='#333',
        font=dict(color='white'),
        bordercolor='#666',
        borderwidth=1
    )],
    title=f'Books about {war_options[0]} by country of author',
    coloraxis=dict(
        colorscale='Viridis',
        cmin=0,
        cmax=war_max[war_options[0]],   # initial max
        colorbar=dict(
            title=dict(
                text='Number of books',
                font=dict(color='#ccc')
            ),
            tickfont=dict(color='#ccc')
        )
    ),
    geo=dict(
        domain=dict(x=[0, 1], y=[0, 1]),
        showframe=False,
        showcoastlines=True,
        coastlinecolor='#555',
        showland=True,
        landcolor='#2a2a3e',
        showcountries=True,
        countrycolor='#555',
        showocean=True,
        oceancolor='#0a0a1a',
        projection_type='equirectangular',
        bgcolor='#1a1a2e'
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#ccc'),
    height=850,
    margin=dict(t=90, l=0, r=0, b=0)
)

# Save to HTML
output_dir = '../frontend/public/graphs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, '7_civil_wars_choropleth.html')
fig11.write_html(output_path)

conn.close()

In [22]:
fig11.show()